# Activation → PCA

Three steps and nothing else:

1. open the activation shards for a selection,
2. filter to a `{cohort: [subjects]}` dictionary you edit,
3. PCA over parcels.

PCA is `numpy.linalg.svd` on the centred matrix — no scikit-learn, so this runs
in the container as shipped.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import nbtools as nb

pd.set_option("display.width", 160)
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False

ATLAS = "yeo7"          # one atlas per read: 111 / 14 / 7 parcels, different schemas
GOOD_FRAMES_ONLY = True  # drop censored TRs
STANDARDIZE = True       # z-score each parcel before PCA -- see the note below
N_COMPONENTS = 10

ROOT = nb.output_root()
print("output_root:", ROOT)

## 1. The selection

`SELECT` is the dictionary. `None` for a cohort means every subject it has.
Leave a cohort out entirely to exclude it.

Run the next cell first if you want to see what is available before choosing.

In [ ]:
inv = nb.inventory("activation")
inv[inv["atlas"] == ATLAS].groupby("cohort").agg(
    subs=("sub", "nunique"), tasks=("task", "nunique"), shards=("path", "size"))

In [ ]:
# EDIT ME. {cohort: [sub, ...]} or {cohort: None} for all of them.
SELECT = {
    cohort: None
    for cohort in sorted(inv.loc[inv["atlas"] == ATLAS, "cohort"].unique())
}

# Resolve against what is on disk, and say loudly if a named subject is absent
# -- a typo would otherwise just shrink the sample silently.
selected, missing = {}, []
for cohort, subs in SELECT.items():
    have = set(inv.loc[(inv["atlas"] == ATLAS) & (inv["cohort"] == cohort), "sub"])
    if not have:
        missing.append(f"{cohort} (cohort has no {ATLAS} shards)")
        continue
    want = have if subs is None else set(map(str, subs))
    missing += [f"{cohort}/{s}" for s in sorted(want - have)]
    selected[cohort] = sorted(want & have)

if missing:
    print("NOT FOUND:", missing)
print({c: len(s) for c, s in selected.items()},
      "->", sum(len(s) for s in selected.values()), "subject(s)")

## 2. Open the shards

In [ ]:
# One read per cohort, pruned to the selected subjects. Partition filters mean
# only those directories are opened.
frames = [nb.load_activation(ATLAS, cohort=cohort, sub=subs, with_parcels=True)
          for cohort, subs in selected.items() if subs]
act = pd.concat(frames, ignore_index=True)
print(f"{len(act):,} rows, {nb.mem_mb(act):.1f} MB")
act.head(3)

In [ ]:
parcels = [c for c in act.columns
           if c not in nb.ACTIVATION_META_COLUMNS
           and c not in ("atlas", "cohort", "task", "sub")]

X = act[parcels]
rows = pd.Series(True, index=act.index)
if GOOD_FRAMES_ONLY:
    rows &= act["good_frame"].astype(bool)

# A parcel that is all-NaN for any selected subject is empty under that
# subject's mask. Dropping the parcel keeps every subject; dropping the rows
# would keep the parcel and lose the subject. Neither is free -- this picks the
# one that preserves n.
empty = [p for p in parcels if X.loc[rows, p].isna().all()
         or act.loc[rows].groupby("sub")[p].apply(lambda s: s.isna().all()).any()]
keep_parcels = [p for p in parcels if p not in empty]
rows &= X[keep_parcels].notna().all(axis=1)

print(f"parcels: {len(parcels)} -> {len(keep_parcels)}"
      + (f"  (dropped {empty})" if empty else ""))
print(f"rows:    {len(act):,} -> {int(rows.sum()):,}")

## 3. PCA

Centring is mandatory. Standardising (z-scoring each parcel) is a choice:
`standardize: false` in every config, so parcels arrive on their own scales and
without it a single high-variance parcel can dominate PC1. Set
`STANDARDIZE = False` above to run on the covariance matrix instead.

In [ ]:
# copy=True because an Arrow-backed frame hands back a read-only view,
# and the centring below writes in place.
M = act.loc[rows, keep_parcels].to_numpy(dtype=np.float64, copy=True)
labels = act.loc[rows, ["cohort", "task", "sub", "t", "time_s"]].reset_index(drop=True)

M -= M.mean(axis=0)
if STANDARDIZE:
    sd = M.std(axis=0, ddof=1)
    sd[sd == 0] = 1.0
    M /= sd

U, S, Vt = np.linalg.svd(M, full_matrices=False)
explained = S**2 / (len(M) - 1)
ratio = explained / explained.sum()

k = min(N_COMPONENTS, len(S))
scores = U[:, :k] * S[:k]                       # rows x k
loadings = pd.DataFrame(Vt[:k].T, index=keep_parcels,
                        columns=[f"PC{i+1}" for i in range(k)])

print(f"{M.shape[0]:,} rows x {M.shape[1]} parcels")
pd.DataFrame({"explained_var_ratio": ratio[:k].round(4),
              "cumulative": ratio[:k].cumsum().round(4)},
             index=[f"PC{i+1}" for i in range(k)])

In [ ]:
if HAVE_MPL:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].bar(range(1, k + 1), ratio[:k])
    ax[0].set_xlabel("component"); ax[0].set_ylabel("explained variance ratio")
    im = ax[1].imshow(loadings.T, aspect="auto", cmap="RdBu_r",
                      vmin=-np.abs(loadings.values).max(),
                      vmax=np.abs(loadings.values).max())
    ax[1].set_yticks(range(k)); ax[1].set_yticklabels(loadings.columns, fontsize=7)
    ax[1].set_xticks(range(len(keep_parcels)))
    ax[1].set_xticklabels(keep_parcels, rotation=90, fontsize=6)
    ax[1].set_title("loadings", fontsize=9)
    fig.colorbar(im, ax=ax[1], shrink=0.8)
    fig.tight_layout()

In [ ]:
# Loadings: what each component is made of.
loadings.round(3)

In [ ]:
# Scores, back on the rows they came from -- so a component can be looked at
# per cohort, per subject, or over stimulus time.
out = pd.concat([labels, pd.DataFrame(scores, columns=loadings.columns)], axis=1)
print(out.groupby("cohort")[list(loadings.columns[:4])].std().round(3)
        .rename(columns=lambda c: f"{c}_sd"))
out.head()

## Notes

* **Rows are TRs pooled across subjects**, so PC1 is the dominant spatial
  pattern across the whole selection, not a per-subject one. For per-subject
  PCA, loop `selected` and run the cell above inside the loop.
* **Pooling cohorts pools acquisitions.** Different TR, bandpass and echo count
  across cohorts (see `nb.cohort_configs()`), so a component that separates
  cohorts may be separating scanners. Selecting one cohort avoids the question
  entirely.
* Switch `ATLAS` to `"harvardoxford"` for 111 parcels — one cohort at a time,
  and check `nb.estimate_gb` first if the selection is large.